# Validate Silver project-to-region mapping

## Purpose

Validate Table 13 row preservation, deterministic identity, matching precedence, official PSGC targets, Central Office handling, boundary quality, conflict visibility, lineage, and coverage. Evidence is persisted before the blocking gate. The validator reports all seven project data-quality attributes and never changes mapping decisions.


## Step 1 — Calculate one grouped metric set

The grouped scan protects the output grain and computes both blocking defects and review findings. Coverage uses the complete project count as its denominator. Boundary diagnostics use the selected 17-row region subset for spatial readiness while separately retaining full-source lineage duplicates as a flag.


In [0]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `04-validation`;

CREATE OR REPLACE TEMPORARY VIEW silver_project_region_map_metrics AS
WITH project_metrics AS (
    SELECT COUNT(*) AS project_rows FROM `02-silver`.silver_project
),
map_metrics AS (
    SELECT
        COUNT(*) AS map_rows,
        COUNT_IF(contract_id IS NOT NULL AND TRIM(contract_id) <> '')
          - COUNT(DISTINCT CASE WHEN contract_id IS NOT NULL AND TRIM(contract_id) <> ''
              THEN NAMED_STRUCT(
                  'source_system', source_system, 'contract_id', contract_id, 'run_id', run_id
              )
            END) AS duplicate_grain_rows,
        COUNT(*) - COUNT(DISTINCT project_region_map_key) AS duplicate_map_keys,
        COUNT_IF(match_status NOT IN (
            'MATCHED', 'MATCHED_WITH_CONFLICT', 'NON_GEOGRAPHIC',
            'AMBIGUOUS', 'UNMATCHED', 'INVALID_SOURCE'
        ) OR match_status IS NULL) AS invalid_match_statuses,
        COUNT_IF(mapping_method NOT IN (
            'MANUAL_OVERRIDE', 'EXACT_REGION_NAME',
            'APPROVED_REGION_ALIAS', 'COORDINATE_BOUNDARY',
            'NON_GEOGRAPHIC_RULE', 'NONE'
        ) OR mapping_method IS NULL) AS invalid_mapping_methods,
        COUNT_IF(match_quality NOT IN (
            'REVIEWED', 'EXACT', 'APPROVED_ALIAS', 'SPATIAL',
            'REVIEW_REQUIRED', 'UNRESOLVED'
        ) OR match_quality IS NULL) AS invalid_match_quality,
        COUNT_IF(reported_region_standardized = 'CENTRAL OFFICE'
            AND (match_status <> 'NON_GEOGRAPHIC'
                 OR mapping_method <> 'NON_GEOGRAPHIC_RULE'
                 OR psgc_region_code IS NOT NULL)) AS invalid_central_office_rows,
        COUNT_IF(
            (reported_region_standardized <> 'CENTRAL OFFICE' AND manual_candidate_count > 0
             AND mapping_method <> 'MANUAL_OVERRIDE')
         OR (reported_region_standardized <> 'CENTRAL OFFICE' AND manual_candidate_count = 0
             AND exact_candidate_count > 0 AND mapping_method <> 'EXACT_REGION_NAME')
         OR (reported_region_standardized <> 'CENTRAL OFFICE' AND manual_candidate_count = 0
             AND exact_candidate_count = 0 AND alias_candidate_count > 0
             AND mapping_method <> 'APPROVED_REGION_ALIAS')
         OR (reported_region_standardized <> 'CENTRAL OFFICE' AND manual_candidate_count = 0
             AND exact_candidate_count = 0 AND alias_candidate_count = 0
             AND spatial_candidate_count > 0 AND mapping_method <> 'COORDINATE_BOUNDARY')
         OR (mapping_method = 'COORDINATE_BOUNDARY'
             AND (manual_candidate_count > 0 OR exact_candidate_count > 0 OR alias_candidate_count > 0))
        ) AS precedence_violations,
        COUNT_IF(
            (match_status = 'MATCHED_WITH_CONFLICT' AND coordinate_crosscheck_status <> 'REGION_MISMATCH')
         OR (coordinate_crosscheck_status = 'REGION_MISMATCH' AND match_status <> 'MATCHED_WITH_CONFLICT')
        ) AS invalid_conflict_states,
        COUNT_IF(match_status IN ('MATCHED', 'MATCHED_WITH_CONFLICT')
            AND (psgc_region_code IS NULL OR psgc_region_name IS NULL)) AS missing_matched_targets,
        COUNT_IF(match_status NOT IN ('MATCHED', 'MATCHED_WITH_CONFLICT')
            AND psgc_region_code IS NOT NULL) AS targets_on_unresolved_rows,
        COUNT_IF(
            source_system IS NULL OR project_key IS NULL
            OR project_source_snapshot_id IS NULL OR project_source_ingest_run_id IS NULL
            OR project_run_id IS NULL OR project_rule_version IS NULL
            OR psgc_source_snapshot_id IS NULL OR psgc_source_ingest_run_id IS NULL
            OR psgc_run_id IS NULL OR psgc_rule_version IS NULL
            OR psgc_source_load_ts IS NULL
            OR boundary_source_snapshot_id IS NULL OR boundary_source_ingest_run_id IS NULL
            OR boundary_source_load_ts IS NULL
            OR mapping_rule_version IS NULL OR run_id IS NULL
            OR source_load_ts IS NULL
        ) AS missing_mandatory_lineage,
        COUNT_IF(project_source_version IS NULL OR TRIM(project_source_version) = '')
            AS missing_optional_project_version,
        COUNT_IF(boundary_source_version IS NULL OR TRIM(boundary_source_version) = '')
            AS missing_optional_boundary_version,
        COUNT_IF(mapping_method = 'MANUAL_OVERRIDE' AND (
            manual_target_psgc_code IS NULL OR manual_override_reason IS NULL
            OR manual_source_reference IS NULL
        )) AS missing_manual_override_lineage,
        COUNT_IF(project_region_map_key <> SHA2(CONCAT_WS('|', source_system, project_key, run_id), 256))
            AS invalid_deterministic_keys,
        COUNT_IF(run_id <> SHA2(CONCAT_WS(
            '||', project_run_id, project_source_snapshot_id,
            psgc_run_id, psgc_source_snapshot_id, COALESCE(psgc_version, 'NO_PUBLISHER_VERSION'),
            boundary_source_snapshot_id, COALESCE(boundary_version, 'NO_BOUNDARY_VERSION'),
            boundary_source_ingest_run_id, alias_version, manual_mapping_version,
            mapping_rule_version
        ), 256)) AS invalid_deterministic_runs,
        COUNT(DISTINCT run_id) AS map_run_count,
        COUNT_IF(match_status = 'MATCHED') AS matched_rows,
        COUNT_IF(match_status = 'MATCHED_WITH_CONFLICT') AS conflict_rows,
        COUNT_IF(match_status = 'NON_GEOGRAPHIC') AS non_geographic_rows,
        COUNT_IF(match_status = 'AMBIGUOUS') AS ambiguous_rows,
        COUNT_IF(match_status = 'UNMATCHED') AS unmatched_rows,
        COUNT_IF(match_status = 'INVALID_SOURCE') AS invalid_source_rows,
        COUNT_IF(reported_region_raw IS NOT NULL AND TRIM(reported_region_raw) <> '')
            AS usable_reported_region_rows,
        COUNT_IF(mapping_method = 'MANUAL_OVERRIDE') AS manual_override_rows,
        COUNT_IF(mapping_method = 'EXACT_REGION_NAME') AS exact_region_name_rows,
        COUNT_IF(mapping_method = 'APPROVED_REGION_ALIAS') AS approved_alias_match_rows,
        COUNT_IF(mapping_method = 'COORDINATE_BOUNDARY') AS coordinate_fallback_rows,
        COUNT_IF(exact_candidate_count > 1 OR alias_candidate_count > 1)
            AS ambiguous_text_rows,
        COUNT_IF(reported_region_raw IS NOT NULL AND TRIM(reported_region_raw) <> ''
            AND manual_candidate_count = 0 AND exact_candidate_count = 0
            AND alias_candidate_count = 0) AS unresolved_text_rows,
        COUNT_IF(reported_region_raw IS NULL OR TRIM(reported_region_raw) = '') AS missing_reported_region_rows,
        COUNT_IF(coordinate_status = 'VALID_PAIR') AS usable_coordinate_rows,
        COUNT_IF(coordinate_status = 'MISSING_PAIR') AS missing_coordinate_rows,
        COUNT_IF(coordinate_status IN ('PARTIAL_PAIR', 'OUTSIDE_SCREENING_RANGE'))
            AS invalid_coordinate_rows,
        COUNT_IF(coordinate_status = 'VALID_PAIR' AND spatial_candidate_count = 0
            AND reported_region_standardized <> 'CENTRAL OFFICE') AS no_boundary_match_rows,
        COUNT_IF(spatial_candidate_count = 1) AS coordinate_spatial_match_rows,
        COUNT_IF(spatial_candidate_count > 1) AS multiple_spatial_candidate_rows,
        COUNT_IF(coordinate_crosscheck_status = 'REGION_MATCH') AS coordinate_agreement_rows,
        COUNT_IF(coordinate_crosscheck_status = 'REGION_MISMATCH') AS coordinate_conflict_rows,
        COUNT_IF(coordinate_crosscheck_status IN (
            'NOT_AVAILABLE', 'NO_BOUNDARY_MATCH', 'AMBIGUOUS_BOUNDARY'
        )) AS unresolved_coordinate_comparison_rows,
        COUNT_IF(reported_region_standardized = 'CENTRAL OFFICE'
            AND manual_candidate_count > 0) AS central_office_override_rows,
        COUNT_IF(alias_version = 'NO_APPROVED_ALIAS_VERSION') AS rows_without_alias_version,
        COUNT_IF(manual_mapping_version = 'NO_APPROVED_MANUAL_VERSION') AS rows_without_manual_version,
        COUNT_IF(boundary_version_status = 'VERSION_MISMATCH_VISIBLE') AS boundary_version_mismatch_rows,
        MAX(run_id) AS pipeline_run_id, MAX(project_source_snapshot_id) AS source_snapshot_id,
        MAX(mapping_rule_version) AS rule_version
    FROM `02-silver`.silver_project_region_map
),
target_metrics AS (
    SELECT COUNT(*) AS invalid_target_rows
    FROM `02-silver`.silver_project_region_map AS mapping
    LEFT ANTI JOIN `02-silver`.silver_psgc_place AS region
      ON mapping.psgc_region_code = region.psgc_code
     AND mapping.psgc_source_snapshot_id = region.source_snapshot_id
     AND region.place_type = 'REGION'
    WHERE mapping.psgc_region_code IS NOT NULL
),
mapping_context AS (
    SELECT MAX(boundary_source_snapshot_id) AS boundary_source_snapshot_id,
           MAX(psgc_source_snapshot_id) AS psgc_source_snapshot_id
    FROM `02-silver`.silver_project_region_map
),
boundary_metrics AS (
    SELECT
        COUNT(*) AS selected_region_boundary_rows,
        COUNT_IF(region.psgc_code IS NOT NULL) AS resolved_boundary_psgc_rows,
        COUNT_IF(region.psgc_code IS NULL) AS unresolved_boundary_psgc_rows,
        COUNT_IF(TRY_TO_GEOMETRY(REGEXP_REPLACE(
            REGEXP_REPLACE(
                boundary.geometry_json, '\"(\\[[^\"]*\\])\"', CONCAT(CHAR(36), '1')
            ),
            '\"(-?[0-9]+(?:\\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?)\"', CONCAT(CHAR(36), '1')
        )) IS NULL) AS invalid_region_geometry_rows,
        COUNT(*) - COUNT(DISTINCT boundary.psgc_code) AS duplicate_region_boundary_rows
    FROM `01-bronze`.boundaries AS boundary
    CROSS JOIN mapping_context AS context
    LEFT JOIN `02-silver`.silver_psgc_place AS region
      ON boundary.psgc_code = region.psgc_code
     AND region.source_snapshot_id = context.psgc_source_snapshot_id
     AND region.place_type = 'REGION'
    WHERE boundary._source_snapshot_id = context.boundary_source_snapshot_id
      AND LOWER(TRIM(boundary.administrative_level)) = 'region'
),
full_boundary_metrics AS (
    SELECT COUNT(*) AS boundary_rows,
        COUNT(*) - COUNT(DISTINCT NAMED_STRUCT(
            'source_file', source_file, 'source_feature_id', source_feature_id
        )) AS duplicate_boundary_lineage_rows
    FROM `01-bronze`.boundaries AS boundary
    CROSS JOIN mapping_context AS context
    WHERE boundary._source_snapshot_id = context.boundary_source_snapshot_id
),
region_metrics AS (
    SELECT COUNT(*) AS official_region_rows
    FROM `02-silver`.silver_psgc_place WHERE place_type = 'REGION'
),
config_metrics AS (
    SELECT
      (SELECT COUNT(*) FROM `02-silver`.config_place_name_alias
       WHERE source_system = 'dpwh_projects' AND UPPER(TRIM(place_type)) = 'REGION'
         AND approval_status = 'APPROVED' AND is_active = TRUE) AS approved_alias_rows,
      (SELECT COUNT(*) FROM `02-silver`.config_manual_geographic_match
       WHERE source_system = 'DPWH' AND UPPER(TRIM(record_type)) = 'PROJECT'
         AND approval_status = 'APPROVED' AND is_active = TRUE) AS approved_manual_rows
)
SELECT * FROM project_metrics
CROSS JOIN map_metrics
CROSS JOIN target_metrics
CROSS JOIN boundary_metrics
CROSS JOIN full_boundary_metrics
CROSS JOIN region_metrics
CROSS JOIN config_metrics;


## Step 2 — Express STOP and FLAG checks across all seven attributes

STOP checks protect row accounting, uniqueness, controlled states, precedence, official targets, Central Office, deterministic identities, and mandatory lineage. FLAG checks retain real source and configuration limitations with explicit denominators.


In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW silver_project_region_map_checks AS
WITH m AS (SELECT * FROM silver_project_region_map_metrics),
checks AS (
    SELECT 'Completeness' AS attribute, 'Every canonical project has one mapping result' AS check_name,
        ABS(map_rows - project_rows) AS failed_rows, project_rows AS total_rows, 'stop' AS action,
        CONCAT('Projects=', project_rows, '; map rows=', map_rows) AS details FROM m
    UNION ALL SELECT 'Uniqueness', 'Usable natural mapping grain is unique', duplicate_grain_rows, map_rows, 'stop',
        CONCAT('Duplicate usable source-system, Contract-ID, run rows=', duplicate_grain_rows) FROM m
    UNION ALL SELECT 'Uniqueness', 'Deterministic map keys are unique', duplicate_map_keys, map_rows, 'stop',
        CONCAT('Duplicate map keys=', duplicate_map_keys) FROM m
    UNION ALL SELECT 'Auditability', 'Map keys reproduce from stable project and run identity',
        invalid_deterministic_keys, map_rows, 'stop', CONCAT('Invalid deterministic keys=', invalid_deterministic_keys) FROM m
    UNION ALL SELECT 'Timeliness', 'Mapping run identity is deterministic and singular',
        invalid_deterministic_runs + ABS(map_run_count - 1), map_rows, 'stop',
        CONCAT('Invalid run identities=', invalid_deterministic_runs, '; distinct runs=', map_run_count) FROM m
    UNION ALL SELECT 'Validity', 'Match statuses use the controlled contract', invalid_match_statuses, map_rows, 'stop',
        CONCAT('Invalid statuses=', invalid_match_statuses) FROM m
    UNION ALL SELECT 'Validity', 'Mapping methods use the controlled contract', invalid_mapping_methods, map_rows, 'stop',
        CONCAT('Invalid methods=', invalid_mapping_methods) FROM m
    UNION ALL SELECT 'Validity', 'Match quality uses the controlled contract', invalid_match_quality, map_rows, 'stop',
        CONCAT('Invalid quality values=', invalid_match_quality) FROM m
    UNION ALL SELECT 'Accuracy', 'Accepted targets are selected official PSGC regions', invalid_target_rows, map_rows, 'stop',
        CONCAT('Invalid accepted targets=', invalid_target_rows) FROM m
    UNION ALL SELECT 'Accuracy', 'Matched rows have one target and unresolved rows have none',
        missing_matched_targets + targets_on_unresolved_rows, map_rows, 'stop',
        CONCAT('Missing matched targets=', missing_matched_targets, '; targets on unresolved rows=', targets_on_unresolved_rows) FROM m
    UNION ALL SELECT 'Consistency', 'Automatic matching follows the documented precedence', precedence_violations, map_rows, 'stop',
        CONCAT('Precedence violations=', precedence_violations) FROM m
    UNION ALL SELECT 'Consistency', 'Final statuses account for every mapping row',
        ABS(map_rows - matched_rows - conflict_rows - non_geographic_rows - ambiguous_rows - unmatched_rows - invalid_source_rows),
        map_rows, 'stop', CONCAT('Matched=', matched_rows, '; conflicts=', conflict_rows, '; non-geographic=', non_geographic_rows,
        '; ambiguous=', ambiguous_rows, '; unmatched=', unmatched_rows, '; invalid=', invalid_source_rows) FROM m
    UNION ALL SELECT 'Consistency', 'Name-versus-coordinate conflicts remain explicit', invalid_conflict_states, map_rows, 'stop',
        CONCAT('Invalid conflict states=', invalid_conflict_states) FROM m
    UNION ALL SELECT 'Validity', 'Central Office stays non-geographic without a PSGC region',
        invalid_central_office_rows, map_rows, 'stop',
        CONCAT('Incorrect Central Office rows=', invalid_central_office_rows) FROM m
    UNION ALL SELECT 'Auditability', 'Required project, PSGC, boundary, config, rule, and load lineage is present',
        missing_mandatory_lineage, map_rows, 'stop', CONCAT('Rows missing mandatory lineage=', missing_mandatory_lineage) FROM m
    UNION ALL SELECT 'Auditability', 'Applied manual overrides retain reviewed decision evidence',
        missing_manual_override_lineage, manual_override_rows, 'stop',
        CONCAT('Applied manual overrides missing target, reason, or source reference=',
               missing_manual_override_lineage) FROM m
    UNION ALL SELECT 'Validity', 'Selected region boundary geometry parses safely', invalid_region_geometry_rows,
        selected_region_boundary_rows, 'stop', CONCAT('Invalid region geometries=', invalid_region_geometry_rows) FROM m
    UNION ALL SELECT 'Uniqueness', 'Selected region boundary PSGC codes are unique',
        duplicate_region_boundary_rows, selected_region_boundary_rows, 'stop',
        CONCAT('Duplicate region boundary codes=', duplicate_region_boundary_rows) FROM m
    UNION ALL SELECT 'Completeness', 'Geographic match coverage is reported against all projects',
        ambiguous_rows + unmatched_rows + invalid_source_rows, map_rows, 'flag',
        CONCAT('Denominator=', map_rows, '; matched=', matched_rows + conflict_rows, '; non-geographic=', non_geographic_rows,
        '; unresolved=', ambiguous_rows + unmatched_rows + invalid_source_rows) FROM m
    UNION ALL SELECT 'Completeness', 'Missing reported regions remain visible', missing_reported_region_rows, map_rows, 'flag',
        CONCAT('Missing reported region=', missing_reported_region_rows, '; denominator=', map_rows) FROM m
    UNION ALL SELECT 'Completeness', 'Reported-region matching coverage remains visible',
        unresolved_text_rows, usable_reported_region_rows, 'flag',
        CONCAT('Usable reported region=', usable_reported_region_rows, '; manual=', manual_override_rows,
        '; exact=', exact_region_name_rows, '; approved alias=', approved_alias_match_rows,
        '; unresolved text=', unresolved_text_rows, '; ambiguous text=', ambiguous_text_rows) FROM m
    UNION ALL SELECT 'Completeness', 'Missing coordinate pairs remain visible', missing_coordinate_rows, map_rows, 'flag',
        CONCAT('Missing coordinate pairs=', missing_coordinate_rows, '; denominator=', map_rows) FROM m
    UNION ALL SELECT 'Validity', 'Partial and outside-screening coordinates remain visible', invalid_coordinate_rows, map_rows, 'flag',
        CONCAT('Coordinate exceptions=', invalid_coordinate_rows, '; denominator=', map_rows) FROM m
    UNION ALL SELECT 'Completeness', 'Usable coordinates without a region boundary match remain visible',
        no_boundary_match_rows, map_rows, 'flag', CONCAT('No boundary match=', no_boundary_match_rows, '; denominator=', map_rows) FROM m
    UNION ALL SELECT 'Completeness', 'Coordinate fallback coverage remains visible',
        no_boundary_match_rows + multiple_spatial_candidate_rows, usable_coordinate_rows, 'flag',
        CONCAT('Usable coordinates=', usable_coordinate_rows, '; one-region spatial matches=',
        coordinate_spatial_match_rows, '; coordinate fallback accepted=', coordinate_fallback_rows,
        '; no boundary match=', no_boundary_match_rows, '; multiple-region candidates=',
        multiple_spatial_candidate_rows) FROM m
    UNION ALL SELECT 'Completeness', 'Approved region-alias availability remains visible',
        CASE WHEN approved_alias_rows = 0 THEN 1 ELSE 0 END, 1, 'flag',
        CONCAT('Approved DPWH region aliases=', approved_alias_rows, '; rows carrying no-alias sentinel=', rows_without_alias_version) FROM m
    UNION ALL SELECT 'Auditability', 'Approved manual-exception availability remains visible',
        CASE WHEN approved_manual_rows = 0 THEN 1 ELSE 0 END, 1, 'flag',
        CONCAT('Approved DPWH project overrides=', approved_manual_rows, '; rows carrying no-manual sentinel=', rows_without_manual_version) FROM m
    UNION ALL SELECT 'Accuracy', 'Name-versus-coordinate region conflicts are reported', conflict_rows, map_rows, 'flag',
        CONCAT('Agreement=', coordinate_agreement_rows, '; conflicts=', coordinate_conflict_rows,
        '; unresolved comparisons=', unresolved_coordinate_comparison_rows, '; denominator=', map_rows) FROM m
    UNION ALL SELECT 'Consistency', 'Central Office override conflicts remain visible',
        central_office_override_rows, non_geographic_rows, 'flag',
        CONCAT('Central Office rows with an approved override=', central_office_override_rows,
        '; non-geographic denominator=', non_geographic_rows) FROM m
    UNION ALL SELECT 'Consistency', 'Boundary PSGC-version differences remain visible',
        boundary_version_mismatch_rows, map_rows, 'flag',
        CONCAT('Rows with boundary/PSGC version mismatch=', boundary_version_mismatch_rows, '; denominator=', map_rows) FROM m
    UNION ALL SELECT 'Completeness', 'Region boundary coverage is reported against official PSGC regions',
        GREATEST(official_region_rows - resolved_boundary_psgc_rows, 0), official_region_rows, 'flag',
        CONCAT('Boundary regions=', selected_region_boundary_rows, '; resolved current PSGC regions=',
        resolved_boundary_psgc_rows, '; official regions=', official_region_rows) FROM m
    UNION ALL SELECT 'Validity', 'Unresolved boundary PSGC codes remain visible',
        unresolved_boundary_psgc_rows, selected_region_boundary_rows, 'flag',
        CONCAT('Unresolved boundary PSGC rows=', unresolved_boundary_psgc_rows,
        '; selected region boundaries=', selected_region_boundary_rows) FROM m
    UNION ALL SELECT 'Uniqueness', 'Full boundary source lineage duplicates remain visible',
        duplicate_boundary_lineage_rows, boundary_rows, 'flag',
        CONCAT('Duplicate source-file/feature rows=', duplicate_boundary_lineage_rows) FROM m
    UNION ALL SELECT 'Completeness', 'Ambiguous projects remain visible', ambiguous_rows, map_rows, 'flag',
        CONCAT('Ambiguous projects=', ambiguous_rows, '; denominator=', map_rows) FROM m
    UNION ALL SELECT 'Completeness', 'Unmatched projects remain visible', unmatched_rows, map_rows, 'flag',
        CONCAT('Unmatched projects=', unmatched_rows, '; denominator=', map_rows) FROM m
    UNION ALL SELECT 'Auditability', 'Optional DPWH publisher version gaps remain visible',
        missing_optional_project_version, map_rows, 'flag',
        CONCAT('Rows without publisher source version=', missing_optional_project_version, '; snapshot lineage remains mandatory') FROM m
    UNION ALL SELECT 'Auditability', 'Optional boundary publisher version gaps remain visible',
        missing_optional_boundary_version, map_rows, 'flag',
        CONCAT('Rows without boundary publisher version=', missing_optional_boundary_version,
        '; boundary snapshot lineage remains mandatory') FROM m
),
context AS (
    SELECT pipeline_run_id, source_snapshot_id, rule_version,
           SHA2(CONCAT_WS('|', pipeline_run_id, 'silver-project-region-map-validation-v1'), 256) AS validation_run_id,
           CURRENT_TIMESTAMP() AS run_ts
    FROM m
)
SELECT
    context.validation_run_id, context.pipeline_run_id, 'silver' AS layer,
    'silver_project_region_map' AS table_name, context.source_snapshot_id, context.rule_version,
    checks.attribute AS data_quality_attribute, checks.check_name AS data_quality_check,
    CAST(checks.failed_rows AS BIGINT) AS failed_rows, CAST(checks.total_rows AS BIGINT) AS total_rows,
    CASE WHEN checks.total_rows > 0 THEN ROUND(100.0 * checks.failed_rows / checks.total_rows, 4) END AS percentage,
    CASE WHEN checks.failed_rows = 0 THEN 'PASS' WHEN checks.action = 'stop' THEN 'FAIL' ELSE 'FLAG' END AS status,
    checks.action, checks.details, context.run_ts
FROM checks CROSS JOIN context;


## Step 3 — Persist evidence before enforcing the gate

The deterministic validation identity updates the same evidence on an exact rerun. The `MERGE` completes before `ASSERT_TRUE`, so failed STOP evidence remains reviewable.


In [0]:
%sql
CREATE TABLE IF NOT EXISTS `04-validation`.silver_dq_results (
    validation_run_id STRING NOT NULL, pipeline_run_id STRING NOT NULL, layer STRING NOT NULL,
    table_name STRING NOT NULL, source_snapshot_id STRING, rule_version STRING,
    data_quality_attribute STRING NOT NULL, data_quality_check STRING NOT NULL,
    failed_rows BIGINT, total_rows BIGINT, percentage DOUBLE, status STRING NOT NULL,
    action STRING NOT NULL, details STRING, run_ts TIMESTAMP NOT NULL
)
USING DELTA
COMMENT 'Reusable Silver transformation validation evidence';

MERGE INTO `04-validation`.silver_dq_results AS target
USING silver_project_region_map_checks AS source
ON target.validation_run_id = source.validation_run_id
AND target.table_name = source.table_name
AND target.data_quality_check = source.data_quality_check
WHEN MATCHED THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

SELECT ASSERT_TRUE(
    COUNT_IF(status = 'FAIL' AND action = 'stop') = 0,
    CONCAT(COUNT_IF(status = 'FAIL' AND action = 'stop'),
           ' blocking Silver project-region mapping checks failed')
)
FROM silver_project_region_map_checks;


## Summary

Validation protects the exact Table 13 grain, project-row preservation, deterministic key and run identity, precedence, official region targets, Central Office exclusion, conflict visibility, boundary quality, and complete lineage. Coverage and source limitations remain non-blocking flags with denominators. Results are stored before the gate. Requires Databricks execution.


In [0]:
%sql
SELECT validation_run_id, data_quality_attribute, status, action, failed_rows,
       total_rows, percentage, data_quality_check, details
FROM silver_project_region_map_checks
ORDER BY CASE status WHEN 'FAIL' THEN 1 WHEN 'FLAG' THEN 2 ELSE 3 END,
         data_quality_attribute, data_quality_check;
